In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd
from tqdm import tqdm

# Allow more columns to be displayed
pd.set_option("display.max_columns", None)

import logging
logging.basicConfig(level=logging.WARNING)

In [3]:
# Allow imports from parent directory
import sys
sys.path.append('..')
from utils.flood_request_utils import (
    # Hazard
    get_wri_and_si_hazard_data,
    plot_wri_and_si_hazard_data,
    get_wri_and_si_hazard_data_multiple,

    # Vulnerability
    plot_wri_and_si_vulnerability_data,
    apply_damage_fraction,
    get_damage_fraction
)

In [4]:
!which python

/Users/klemenkubelj/miniconda3/envs/cvar-masters/bin/python


In [5]:
DATA_DIR = "../data"

In [6]:
# Define the column names we want to check
predicted_columns = [
    "predicted_wri_depth",
    "predicted_si_depth",
    "predicted_si_depth_100r",
    "predicted_si_depth_100r_max",
    "predicted_si_depth_1000r",
    "predicted_si_depth_1000r_max",
    "predicted_si_old_depth",
    "predicted_wri_damage",
    "predicted_si_damage",
    "predicted_si_old_damage"
]

In [11]:
vloge_df = pd.read_csv(os.path.join(DATA_DIR, "vloge_processed_with_gps_filtered_2025-05-10.csv"))
vloge_df.shape

(13531, 18)

In [32]:
# NOTE: Comment out this if you want to calculate the flood depths from scratch
cached_df = pd.read_excel(os.path.join("../data/predicted_flood_depths_2025-10-28.xlsx"))
vloge_df = vloge_df.merge(cached_df[
    ["VlogaId"] + predicted_columns
], on=["VlogaId"], how="left")

In [33]:
vloge_df.head()

,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Skoda_DatumOcene,Objekt_VrstaObjektaId,DogodekId,Vrednost,OdstPoskodovanostiObjekta,SkupnaSkoda,SkupnaSkodaSource,geometry,gps_lat,gps_lng,predicted_wri_depth,predicted_si_depth,predicted_si_depth_100r,predicted_si_depth_100r_max,predicted_si_depth_1000r,predicted_si_depth_1000r_max,predicted_si_old_depth,predicted_wri_damage,predicted_si_damage,predicted_si_old_damage
0,147651,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,50.0,30.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126,0.000000,1.5,0.5,1.5,1.5,2.5,0.0,0.000000,0.500,0.0
1,147686,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,55.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,0.000000,0.5,0.5,1.5,1.5,2.5,0.0,0.000000,0.250,0.0
2,147692,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,38.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,0.000000,0.5,0.5,1.5,1.5,2.5,0.0,0.000000,0.250,0.0
3,147704,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,80.0,40.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126,0.000000,1.5,0.5,1.5,1.5,2.5,0.0,0.000000,0.500,0.0
4,147723,Železniška cesta 8,6280 ANKARAN - ANCARANO,877/7,705,290.0,130.0,NaN,10/01/10 00:00:00,8.0,14,NaN,NaN,NaN,NaN,POINT (13.756193519543181 45.562016308567),45.562016,13.756194,0.482443,2.5,2.5,2.5,2.5,2.5,0.0,0.241221,0.675,0.0


In [34]:
# vloge_df.loc[10:15, predicted_columns] = pd.NA
# vloge_df[vloge_df["predicted_wri_depth"].isna()]

In [35]:
# data, request = get_wri_and_si_hazard_data_multiple([
#     {
#         "lat": 45.555094,
#         "lng": 13.786126
#     },
#     {
#         "lat": 45.555094,
#         "lng": 13.786126
#     }
# ])

In [36]:
def calc_predicted_flood_depth_batch(df_subset, batch_size=1000):
    """
    Calculate flood depths for multiple coordinates in batches for better performance.
    
    Args:
        df_subset: DataFrame with rows that need calculation
        batch_size: Number of coordinates to process in each batch
    
    Returns:
        DataFrame with calculated flood depths
    """
    result_df = df_subset.copy()
    
    # Create the columns if they don't exist
    for col in predicted_columns:
        if col not in result_df.columns:
            result_df[col] = None
    
    # Process in batches
    total_rows = len(df_subset)
    print(f"Processing {total_rows} rows in batches of {batch_size}")
    
    for start_idx in tqdm(range(0, total_rows, batch_size), desc="Processing batches"):
        end_idx = min(start_idx + batch_size, total_rows)
        batch_df = df_subset.iloc[start_idx:end_idx]
        
        # Prepare coordinates for batch request
        coords = [
            {"lat": row["gps_lat"], "lng": row["gps_lng"]}
            for _, row in batch_df.iterrows()
        ]
        
        try:
            # Get hazard data for the entire batch
            data, request = get_wri_and_si_hazard_data_multiple(coords)
            
            # Process each coordinate's results
            for i, (_, row) in enumerate(batch_df.iterrows()):
                row_idx = start_idx + i
                
                # Extract flood depths for this coordinate
                wri_depths = data["flood_depths"]["wri"][i]
                si_depths = data["flood_depths"]["si"][i]
                si_res_100_depths = data["flood_depths"]["si_res_100"][i]
                si_res_1000_depths = data["flood_depths"]["si_res_1000"][i]
                si_res_100_max_depths = data["flood_depths"]["si_res_100_max"][i]
                si_res_1000_max_depths = data["flood_depths"]["si_res_1000_max"][i]
                si_old_depths = data["flood_depths"]["si_old"][i]
                
                # Get damage fractions (if available)
                wri_damage = data.get("damage_fractions", {}).get("wri", [{}])[i] if "damage_fractions" in data else {}
                si_damage = data.get("damage_fractions", {}).get("si", [{}])[i] if "damage_fractions" in data else {}
                si_old_damage = data.get("damage_fractions", {}).get("si_old", [{}])[i] if "damage_fractions" in data else {}
                
                # Assign values to result dataframe
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_wri_depth")] = wri_depths.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_depth")] = si_depths.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_depth_100r")] = si_res_100_depths.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_depth_1000r")] = si_res_1000_depths.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_depth_100r_max")] = si_res_100_max_depths.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_depth_1000r_max")] = si_res_1000_max_depths.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_old_depth")] = si_old_depths.get(100, None)
                
                # Damage fractions (if available)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_wri_damage")] = wri_damage.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_damage")] = si_damage.get(100, None)
                result_df.iloc[row_idx, result_df.columns.get_loc("predicted_si_old_damage")] = si_old_damage.get(100, None)
                
        except Exception as e:
            print(f"Error processing batch {start_idx}-{end_idx}: {e}")
            raise
    
    return result_df

# DEPRECATED
# # Original single-row function (kept for reference)
# def calc_predicted_flood_depth(row):
#     # Get the hazard data
#     gps = {
#         "lat": row["gps_lat"],
#         "lng": row["gps_lng"]
#     }
#     data, request = get_wri_and_si_hazard_data(gps)
#     return (
#         data["flood_depths"]["wri"][100], # "predicted_wri_depth",
#         data["flood_depths"]["si"][100], # "predicted_si_depth",
#         data["flood_depths"]["si_res_100"][100], # "predicted_si_depth_100r",
#         data["flood_depths"]["si_res_1000"][100], # "predicted_si_depth_1000r",
#         data["flood_depths"]["si_old"][100], # "predicted_si_old_depth",
#         data["damage_fractions"]["wri"][100], # "predicted_wri_damage",
#         data["damage_fractions"]["si"][100], # "predicted_si_damage",
#         data["damage_fractions"]["si_old"][100], # "predicted_si_old_damage"
#     )

_df = vloge_df.copy()

# Create the columns if they don't exist
for col in predicted_columns:
    if col not in _df.columns:
        _df[col] = None

# Create a mask for rows that need calculation
needs_calculation = _df[predicted_columns].isna().any(axis=1)
print(f"Rows needing calculation: {needs_calculation.sum()}")
# _df = _df[needs_calculation].head(100)

rows_to_calculate = _df[needs_calculation].copy()

# Process in batches (adjust batch_size based on your API limits and memory)
# Start with a smaller batch size to test
batch_size = 300  # You can increase this (e.g., 100, 200) for better performance

print(f"Processing {len(rows_to_calculate)} rows in batches of {batch_size}")
result_df = calc_predicted_flood_depth_batch(rows_to_calculate, batch_size=batch_size)

# Update the original dataframe with the results
_df.update(result_df[predicted_columns])

print("Batch processing completed!")
print(f"Successfully processed: {result_df[predicted_columns].notna().any(axis=1).sum()} rows")

Rows needing calculation: 0
Processing 0 rows in batches of 300
Processing 0 rows in batches of 300


Processing batches: 0it [00:00, ?it/s]

Batch processing completed!
Successfully processed: 0 rows


In [30]:
result_df.head()

,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Skoda_DatumOcene,Objekt_VrstaObjektaId,DogodekId,Vrednost,OdstPoskodovanostiObjekta,SkupnaSkoda,SkupnaSkodaSource,geometry,gps_lat,gps_lng,predicted_wri_depth,predicted_si_depth,predicted_si_depth_100r,predicted_si_depth_100r_max,predicted_si_depth_1000r,predicted_si_depth_1000r_max,predicted_si_old_depth,predicted_wri_damage,predicted_si_damage,predicted_si_old_damage
0,147651,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,50.0,30.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126,0.0,1.5,0.5,1.5,1.5,2.5,0.0,0.0,0.5,0.0
1,147686,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,55.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,0.0,0.5,0.5,1.5,1.5,2.5,0.0,0.0,0.25,0.0
2,147692,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,38.0,50.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861,0.0,0.5,0.5,1.5,1.5,2.5,0.0,0.0,0.25,0.0
3,147704,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,80.0,40.0,NaN,09/30/10 00:00:00,2.0,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126,0.0,1.5,0.5,1.5,1.5,2.5,0.0,0.0,0.5,0.0
4,147723,Železniška cesta 8,6280 ANKARAN - ANCARANO,877/7,705,290.0,130.0,NaN,10/01/10 00:00:00,8.0,14,NaN,NaN,NaN,NaN,POINT (13.756193519543181 45.562016308567),45.562016,13.756194,0.482443,2.5,2.5,2.5,2.5,2.5,0.0,0.241221,0.675,0.0


## Export

In [31]:
result_df["measured_depth"] = result_df["Objekt_VisinaVodeCm"]/100

# Export all cols but "gps_point"
# _df.to_excel(os.path.join("../data/predicted_flood_depths_2025-05-13.xlsx"), index=False)
# result_df.to_excel(os.path.join("../data/predicted_flood_depths_2025-10-26.xlsx"), index=False)
result_df.to_excel(os.path.join("../data/predicted_flood_depths_2025-10-28.xlsx"), index=False)